# NB-303: GPT-Style Decoder

Implement a GPT-style causal language model.

## Learning Objectives
- Causal masking
- Text generation
- Temperature sampling
- Top-k and nucleus sampling

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Ready to build GPT")

## 1. Causal Multi-Head Attention

In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # TODO: Project and reshape
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        # TODO: Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))

        # TODO: Apply causal mask
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        causal_mask = causal_mask.to(x.device)
        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # TODO: Softmax and apply to values
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_o(output)

# Test
mha = CausalMultiHeadAttention(d_model=256, n_heads=4)
x = torch.randn(2, 10, 256)
output = mha(x)
print(f"Output shape: {output.shape}")

## 2. GPT Block

In [ ]:
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        # TODO: Causal attention
        self.attention = CausalMultiHeadAttention(d_model, n_heads, dropout)

        # Feed-forward
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        # Layer norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: Attention with residual
        attn_out = self.attention(x)
        x = x + self.dropout(attn_out)
        x = self.norm1(x)

        # Feed-forward with residual
        ff_out = self.feed_forward(x)
        x = x + self.dropout(ff_out)
        x = self.norm2(x)

        return x

## 3. Complete GPT Model

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=4, d_ff=1024, max_len=512):
        super().__init__()

        # TODO: Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        # TODO: GPT blocks
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])

        # Output head
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        # TODO: Add embeddings
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.pos_embedding(torch.arange(seq_len, device=input_ids.device))
        x = token_emb + pos_emb

        # TODO: Pass through blocks
        for block in self.blocks:
            x = block(x)

        # Output
        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits

# Create model
model = GPT(vocab_size=1000, d_model=128, n_heads=4, n_layers=2)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Text Generation

In [ ]:
def generate(model, input_ids, max_new_tokens=20, temperature=1.0, top_k=None):
    """Generate text with GPT model."""

    model.eval()
    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Forward pass
            logits = model(generated)

            # Get next token logits
            next_token_logits = logits[:, -1, :] / temperature

            # TODO: Apply top-k filtering
            if top_k is not None:
                values, indices = torch.topk(next_token_logits, top_k)
                next_token_logits = torch.full_like(next_token_logits, float('-inf'))
                next_token_logits.scatter_(1, indices, values)

            # Sample
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Append
            generated = torch.cat([generated, next_token], dim=1)

    return generated

# Test generation
prompt_ids = torch.randint(0, 1000, (1, 5))
generated = generate(model, prompt_ids, max_new_tokens=10, temperature=0.8)
print(f"Generated shape: {generated.shape}")

## Exercise

1. Implement nucleus (top-p) sampling
2. Add beam search
3. Implement repetition penalty